# 03 — Train YOLOv10n nhan dien tien VND (Roboflow dataset)

Train YOLOv10n detector tren dataset Roboflow co cac to tien bi:
- gap doi, gap tu
- vo nhau
- ngon tay che 1 phan

**Output:** `vnd_yolov10n.tflite` + `vnd_yolov10n_labels.txt` -> drop vao `app/src/main/assets/ml/`.

**Runtime:** Colab T4 GPU, ~30-90 phut tuy so anh.

## Class mapping
9 classes: `500000, 200000, 100000, 50000, 20000, 10000, 5000, 2000, 1000` (trung thu tu MoneyLabel[0..8] trong Kotlin).

## Cell 1 — Cai dependencies

In [ ]:
!pip install -q ultralytics==8.3.40 roboflow tensorflow onnx onnxslim onnxruntime
import ultralytics
ultralytics.checks()

## Cell 2 — Download Roboflow dataset

Lay API key: https://roboflow.com -> Settings -> API key.

Workspace/project/version: vao Roboflow Universe, search 'vietnamese money' hoac 'VND money'. Vi du dataset:
- `vietnam-money-tdkqd/vnd-money/3` (folded/crumpled bills)

Hoac upload custom dataset cua anh len Roboflow.

In [ ]:
from roboflow import Roboflow

# >>> CHINH SUA 4 BIEN NAY <<<
ROBOFLOW_API_KEY = ''  # paste API key roboflow vao day
WORKSPACE = 'vietnam-money-tdkqd'  # vi du, doi neu can
PROJECT = 'vnd-money'
VERSION = 3

assert ROBOFLOW_API_KEY, 'Dien ROBOFLOW_API_KEY truoc khi chay'
rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace(WORKSPACE).project(PROJECT)
version = project.version(VERSION)
dataset = version.download('yolov8')  # YOLOv10n dung cung format folder voi yolov8
print('Dataset path:', dataset.location)
DATA_YAML = f'{dataset.location}/data.yaml'
!cat {DATA_YAML}

## Cell 3 — Validate class names + map sang VND

Class trong Roboflow co the la: `1000, 2000, ..., 500000` hoac `1k, 2k, ..., 500k` hoac `bill_1000, bill_2000, ...`.

App Kotlin (MONEY_LABELS) expect thu tu: 0=500k, 1=200k, 2=100k, 3=50k, 4=20k, 5=10k, 6=5k, 7=2k, 8=1k.

Cell nay PRINT names hien co + map. Neu khong dung thu tu, cell 4 se REORDER class id qua YAML moi.

In [ ]:
import yaml, re
from pathlib import Path

with open(DATA_YAML) as f:
    data = yaml.safe_load(f)
print('Goc:', data)
names = data['names']
if isinstance(names, dict):
    names = [names[i] for i in sorted(names.keys())]

DESIRED_VND_ORDER = [500000, 200000, 100000, 50000, 20000, 10000, 5000, 2000, 1000]

def parse_vnd(name):
    s = str(name).lower()
    if 'k' in s:
        n = int(re.sub(r'[^0-9]', '', s.split('k')[0]))
        return n * 1000
    return int(re.sub(r'[^0-9]', '', s))

vnd_by_id = {i: parse_vnd(n) for i, n in enumerate(names)}
print('Class id -> VND:', vnd_by_id)
missing = [v for v in DESIRED_VND_ORDER if v not in vnd_by_id.values()]
if missing:
    print('CANH BAO: dataset thieu menh gia:', missing)

## Cell 4 — Re-map class ID theo thu tu mong muon (neu can)

Day du dataset gia su co 9 classes. Tao file YAML moi voi thu tu DESIRED_VND_ORDER + rewrite TXT labels.

In [ ]:
import shutil
from glob import glob

new_id_by_vnd = {v: i for i, v in enumerate(DESIRED_VND_ORDER)}
remap = {}
for old_id, vnd in vnd_by_id.items():
    if vnd in new_id_by_vnd:
        remap[old_id] = new_id_by_vnd[vnd]
print('Remap old_id -> new_id:', remap)

for split in ['train', 'valid', 'test']:
    lbl_dir = Path(dataset.location) / split / 'labels'
    if not lbl_dir.exists():
        continue
    for txt in lbl_dir.glob('*.txt'):
        new_lines = []
        for line in txt.read_text().splitlines():
            parts = line.strip().split()
            if not parts:
                continue
            old = int(parts[0])
            if old not in remap:
                continue
            parts[0] = str(remap[old])
            new_lines.append(' '.join(parts))
        txt.write_text('\n'.join(new_lines) + '\n')

new_yaml = {
    'path': dataset.location,
    'train': 'train/images',
    'val': 'valid/images',
    'test': 'test/images' if (Path(dataset.location) / 'test/images').exists() else None,
    'names': {i: str(v) for i, v in enumerate(DESIRED_VND_ORDER)},
    'nc': len(DESIRED_VND_ORDER),
}
new_yaml_path = '/content/vnd_data.yaml'
with open(new_yaml_path, 'w') as f:
    yaml.safe_dump(new_yaml, f, sort_keys=False)
!cat /content/vnd_data.yaml

## Cell 5 — Train YOLOv10n

100 epochs, imgsz 640, batch tu dong. Resume neu kernel restart.

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov10n.pt')  # pretrained weights

results = model.train(
    data='/content/vnd_data.yaml',
    epochs=100,
    imgsz=640,
    batch=-1,           # auto-batch
    device=0,           # GPU 0
    patience=20,        # early stop
    project='/content/runs',
    name='vnd_yolov10n',
    augment=True,
    mosaic=1.0,
    mixup=0.1,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    fliplr=0.5,
    flipud=0.0,         # tien doc tu thuong khong lat doc, lat doc se sai
    degrees=15.0,       # xoay nho de cover gap/vo
    translate=0.1,
    scale=0.5,
    shear=2.0,
    perspective=0.0,
)

## Cell 6 — Validate

In [ ]:
best_pt = '/content/runs/vnd_yolov10n/weights/best.pt'
model = YOLO(best_pt)
metrics = model.val(data='/content/vnd_data.yaml')
print('mAP50:', metrics.box.map50)
print('mAP50-95:', metrics.box.map)

## Cell 7 — Export TFLite

FP32 export (~5-8 MB). INT8 quant de model nho hon ~2MB nhung accuracy giam, bo qua truoc.

In [ ]:
model = YOLO(best_pt)
tflite_path = model.export(format='tflite', imgsz=640, nms=True, int8=False)
print('TFLite path:', tflite_path)

import shutil
shutil.copy(tflite_path, '/content/vnd_yolov10n.tflite')

# Tao file labels theo thu tu MONEY_LABELS Kotlin (9 classes, KHONG co unknown -- YOLO dung confidence threshold thay vi unknown class)
labels_txt = '\n'.join(str(v) for v in DESIRED_VND_ORDER) + '\n'
with open('/content/vnd_yolov10n_labels.txt', 'w') as f:
    f.write(labels_txt)
print('Labels:')
print(labels_txt)

from pathlib import Path
print('Size:', Path('/content/vnd_yolov10n.tflite').stat().st_size / 1024 / 1024, 'MB')

## Cell 8 — Inspect input/output shape

App Kotlin can biet:
- Input shape (640x640, FP32, range 0..1 hay 0..255?)
- Output shape ([1, N, 6] hay khac?)

In [ ]:
import tensorflow as tf
interp = tf.lite.Interpreter(model_path='/content/vnd_yolov10n.tflite')
interp.allocate_tensors()
print('Inputs:')
for d in interp.get_input_details():
    print(' ', d['name'], d['shape'], d['dtype'])
print('Outputs:')
for d in interp.get_output_details():
    print(' ', d['name'], d['shape'], d['dtype'])

## Cell 9 — Test 1 anh thuc

Verify model chay duoc + xem detection.

In [ ]:
import random
from PIL import Image
val_images = list(Path(dataset.location).glob('valid/images/*.jpg'))
sample = random.choice(val_images)
print('Test:', sample)
res = model.predict(str(sample), conf=0.5, imgsz=640)
for r in res:
    print('Detections:', len(r.boxes))
    for box in r.boxes:
        cls = int(box.cls.item())
        conf = float(box.conf.item())
        vnd = DESIRED_VND_ORDER[cls]
        print(f'  class={cls} -> {vnd} VND, conf={conf:.3f}')

## Cell 10 — Download outputs

Mo Files panel ben trai Colab -> right-click `/content/vnd_yolov10n.tflite` -> Download.
Tuong tu voi `vnd_yolov10n_labels.txt`.

Sau khi tai ve, drop 2 file vao `D:/hotronguoikhiemthi/app/src/main/assets/ml/`.

In [ ]:
from google.colab import files
files.download('/content/vnd_yolov10n.tflite')
files.download('/content/vnd_yolov10n_labels.txt')